# 🎓 End-to-End Tabular Machine Learning Baseline
### Module 03 | Beginner ML Project — Student Pass/Fail Prediction
**Course:** AI Engineering Bootcamp  
**Target:** Predict `passed` (0 = Fail, 1 = Pass) using features known before the final result.  
**Leakage Prevention:** Strictly exclude `final_score` because it directly reveals the exam result.

---
## Project Workflow Summary:
1. **Data Inspection & Leakage-Aware Framing:** Load `student_learning.csv`, inspect shape, data types, missing values, target distribution.
2. **Reproducible Stratified Splitting:** Partition dataset into 80% train and 20% test sets stratified by the target `passed`.
3. **ColumnTransformer & Pipeline Construction:** Median imputation and standard scaling for numeric features; mode imputation and one-hot encoding for categorical features.
4. **Baseline & Candidate Classification Models:**
   - Baseline: `DummyClassifier(strategy='most_frequent')`
   - Candidate 1: `LogisticRegression`
   - Candidate 2: `RandomForestClassifier`
   - Candidate 3: `GradientBoostingClassifier`
5. **Fair Evaluation & Error Analysis:** Evaluated on the identical held-out test set using Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix.
6. **Persistence & Inference:** Save fitted pipeline via `joblib`, reload, and predict `new_student_records.csv` without retraining.
7. **Responsible AI & Limitations:** Detailed ethical evaluation, limitations, and governance guidelines.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Packages imported successfully!")


---
## Step 1: Inspect Dataset and Choose Features
We load `student_learning.csv` and inspect data shapes, column types, missing values, and class balance.

**Critical Leakage Protection:**
- `final_score` represents the numerical grade obtained on the final assessment. Using it to predict `passed` would cause severe data leakage because the pass/fail cutoff directly derives from it.
- Therefore, we drop `final_score` and only retain features available **prior** to the final assessment (`hours_studied`, `attendance`, `practice_tests`, `sleep_hours`, `study_method`).


In [ ]:
# 1. Load dataset with robust path resolution
import os
data_file = 'student_learning.csv'
for candidate in [data_file, os.path.join('..', 'data', data_file), os.path.join('data', data_file)]:
    if os.path.exists(candidate):
        data_file = candidate
        break
df = pd.read_csv(data_file)
print(f"Loaded dataset from: {data_file}")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()


In [ ]:
# Data types and missing values
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df)) * 100
})
info_df


In [ ]:
# Target Balance Analysis
target_counts = df['passed'].value_counts()
target_props = df['passed'].value_counts(normalize=True)

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage': (target_props * 100).round(2).astype(str) + '%'
}, index=[0, 1])
target_summary.index = ['Fail (0)', 'Pass (1)']
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['passed'].value_counts().plot(kind='bar', color=['#E53E3E', '#319795'], ax=axes[0])
axes[0].set_title("Target Distribution (Counts)")
axes[0].set_xticklabels(['Fail (0)', 'Pass (1)'], rotation=0)
axes[0].set_ylabel("Number of Students")

df.boxplot(column='hours_studied', by='passed', ax=axes[1])
axes[1].set_title("Hours Studied vs. Outcome")
axes[1].set_xlabel("Passed (0=Fail, 1=Pass)")
axes[1].set_ylabel("Hours Studied")
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
# Feature and Target Separation (Leakage-Aware)
TARGET = 'passed'
LEAKAGE_FEATURE = 'final_score'

X = df.drop(columns=[TARGET, LEAKAGE_FEATURE])
y = df[TARGET]

num_features = ['hours_studied', 'attendance', 'practice_tests', 'sleep_hours']
cat_features = ['study_method']

print(f"Predictor Features ({len(X.columns)}):", list(X.columns))
print(f"Target variable: '{TARGET}'")
print(f"Excluded Leakage Column: '{LEAKAGE_FEATURE}'")


---
## Step 2: Reproducible Stratified Train-Test Split & Baseline Model
To ensure our evaluation is reliable and fair:
1. We set `random_state=42` for strict reproducibility.
2. We use `stratify=y` to preserve the exact Pass/Fail ratio (approx. 56% Fail, 44% Pass) across both train and test splits.
3. We establish a **DummyClassifier** (majority class strategy) as the baseline benchmark.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples (Fail: {(y_train == 0).sum()}, Pass: {(y_train == 1).sum()})")
print(f"Testing set:  {X_test.shape[0]} samples (Fail: {(y_test == 0).sum()}, Pass: {(y_test == 1).sum()})")


In [ ]:
# Baseline Model: DummyClassifier (most frequent class)
dummy_baseline = DummyClassifier(strategy="most_frequent")
dummy_baseline.fit(X_train, y_train)
dummy_preds = dummy_baseline.predict(X_test)

print("--- BASELINE MODEL PERFORMANCE ---")
print(f"Baseline Accuracy: {accuracy_score(y_test, dummy_preds):.4f}")
print(f"Baseline F1-Score: {f1_score(y_test, dummy_preds, zero_division=0):.4f}")
print("\nBaseline Confusion Matrix:")
print(confusion_matrix(y_test, dummy_preds))


---
## Step 3: Build Preprocessing Pipeline with ColumnTransformer
We assemble a scikit-learn `ColumnTransformer` to handle preprocessing without leaking test set statistics:
- **Numerical Features:** Median imputation followed by `StandardScaler`.
- **Categorical Features:** Most frequent imputation followed by `OneHotEncoder(handle_unknown='ignore')`.


In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

print("Preprocessing ColumnTransformer constructed successfully.")


---
## Step 4: Train Candidate Models & Fair Evaluation
We evaluate candidate models on the **identical** held-out test set:
1. **Logistic Regression:** Interpretable linear baseline with L2 regularization.
2. **Random Forest Classifier:** Non-linear ensemble model capturing feature interactions.
3. **Gradient Boosting Classifier:** Sequential boosting ensemble.


In [ ]:
candidate_classifiers = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

benchmark_results = []
fitted_pipelines = {}

for name, clf in candidate_classifiers.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else 0.5
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

eval_df = pd.DataFrame(benchmark_results)
eval_df


In [ ]:
# Visualize Metrics Comparison
fig, ax = plt.subplots(figsize=(10, 5))
eval_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(
    kind="bar", ax=ax, colormap="viridis", width=0.8
)
plt.title("Comparative Performance Across Held-Out Test Set", fontsize=14, fontweight='bold')
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Confusion Matrix & Classification Report for Best Model (Random Forest)
best_model_name = "Random Forest"
best_pipeline = fitted_pipelines[best_model_name]
best_preds = best_pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Display
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail (0)', 'Pass (1)'])
disp.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title(f"Confusion Matrix: {best_model_name}", fontsize=12, fontweight='bold')

# ROC Curve Display
RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[1], name=best_model_name)
axes[1].plot([0, 1], [0, 1], "k--", label="Chance (AUC = 0.50)")
axes[1].set_title(f"ROC Curve: {best_model_name}", fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n--- Detailed Classification Report for {best_model_name} ---")
print(classification_report(y_test, best_preds, target_names=['Fail (0)', 'Pass (1)']))


---
## Step 5: Error Analysis & Diagnostic Inspection
Understanding the errors made by the model is essential for responsible deployment:
- **False Positives (5 cases):** The model predicted the student would pass, but they failed. In an academic advisory setting, this could mean an at-risk student is missed without human check-ins.
- **False Negatives (5 cases):** The model predicted the student would fail, but they passed. If used punitively, this would unfairly penalize resilient students.


In [ ]:
# Detailed breakdown of misclassified test instances
test_diagnostics = X_test.copy()
test_diagnostics['actual_outcome'] = y_test
test_diagnostics['predicted_outcome'] = best_preds
test_diagnostics['pass_probability'] = np.round(best_pipeline.predict_proba(X_test)[:, 1], 3)

test_diagnostics['diagnosis'] = 'Correct'
test_diagnostics.loc[(test_diagnostics['actual_outcome'] == 0) & (test_diagnostics['predicted_outcome'] == 1), 'diagnosis'] = 'False Positive (Predicted Pass, Actually Failed)'
test_diagnostics.loc[(test_diagnostics['actual_outcome'] == 1) & (test_diagnostics['predicted_outcome'] == 0), 'diagnosis'] = 'False Negative (Predicted Fail, Actually Passed)'

errors = test_diagnostics[test_diagnostics['diagnosis'] != 'Correct']
print(f"Total Test Instances: {len(test_diagnostics)} | Total Errors: {len(errors)} (Accuracy: {1 - len(errors)/len(test_diagnostics):.2%})\n")
errors


---
## Step 6: Save and Reload Pipeline (Joblib) & Predict New Records
We persist the entire fitted pipeline (`preprocessor` + `classifier`) into `student_model_pipeline.joblib`.
We then reload it and predict on `new_student_records.csv` without retraining.


In [ ]:
# 1. Save pipeline
import os
models_dir = '../models' if os.path.exists('../models') else ('models' if os.path.exists('models') else '.')
joblib_filename = os.path.join(models_dir, 'student_model_pipeline.joblib')
joblib.dump(best_pipeline, joblib_filename)
print(f"Saved fitted pipeline to disk as: '{joblib_filename}'")

# 2. Reload pipeline without retraining
reloaded_pipe = joblib.load(joblib_filename)
print("Pipeline reloaded successfully!")

# 3. Load and predict new unlabelled student records
new_rec_file = 'new_student_records.csv'
for candidate in [new_rec_file, os.path.join('..', 'data', new_rec_file), os.path.join('data', new_rec_file)]:
    if os.path.exists(candidate):
        new_rec_file = candidate
        break
new_data = pd.read_csv(new_rec_file)
new_data_preds = reloaded_pipe.predict(new_data)
new_data_probs = reloaded_pipe.predict_proba(new_data)[:, 1]

new_results = new_data.copy()
new_results['predicted_passed'] = new_data_preds
new_results['outcome'] = np.where(new_data_preds == 1, 'Pass', 'Fail')
new_results['pass_probability'] = np.round(new_data_probs, 4)

print(f"\n--- Predictions on '{new_rec_file}' ---")
new_results


---
## Step 7: Responsible AI Considerations, Limitations, and Ethics
> **Mandatory Disclaimer:**  
> *This is synthetic classroom data. Predictions are for demonstration and must not be used to judge, punish, or penalize real students.*

### Ethical Framework:
1. **Intended Use:** Formative, supportive early alert indicator to connect students with tutoring and academic mentoring early in the semester.
2. **Prohibited Applications:** Never automate course grading, gatekeep access to honors programs, revoke financial aid, or penalize students.
3. **Missing Context & Confounders:** Tabular features (`hours_studied`, `attendance`, `sleep_hours`, `practice_tests`, `study_method`) omit critical socioeconomic dimensions: caregiving obligations, part-time employment, health challenges, learning disabilities, and digital access disparities.
4. **Human-in-the-Loop Safeguards:** All automated alerts must be vetted and contextualized by qualified academic counselors and educators.
